# Importar librerías

In [1]:
import sys
from pathlib import Path

# encontrar raíz del proyecto (carpeta que contiene "src")
root = Path().resolve()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))

In [2]:
import pandas as pd
import numpy as np
from src.data.load_data import cargar_csv_con_sniffer
from src.data.clean_preprocess_data import *

# Cargar base de datos

In [3]:
ruta_archivo = '../databases/raw/raw_data_customers (3) (1).csv'

In [4]:
archivo = cargar_csv_con_sniffer(ruta_archivo)
archivo.head()

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
0,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,12/05/2025,450.50,12,0,Calle Falsa 123
1,C002,Maria Garcia,m.garcia@provider.net,555-0102,2023-02-20,2025-05-10,1200.00,45,0,Carrera 7 # 45-10
2,C003,Carlos Rodriguez,c.rod@work.com,,2023-03-05,03/25/2025,NULL,8,1,Av. Siempre Viva 742
3,C004,Ana Martinez,ana.mtz@mail.com,555-0104,2023-04-12,2025-06-01,890.20,22,0,Clle 100 # 15
4,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,12/05/2025,450.50,12,0,Calle Falsa 123


# Estandarizar nulos

In [5]:
archivo = estandarizar_nulos(archivo)

# Arreglar columnas con formato de fecha

In [6]:
archivo = normalizar_columna_fecha(archivo, "signup_date")
archivo = normalizar_columna_fecha(archivo, "last_purchase_date")

# Eliminar duplicados por id

In [7]:
# Hay valores repetidos en customer_id:
archivo['customer_id'].value_counts()

customer_id
C001    2
C003    2
C012    2
C023    2
C002    1
       ..
C106    1
C107    1
C108    1
C109    1
C110    1
Name: count, Length: 110, dtype: int64

In [8]:
# Todos los customer_id tienen una longitud de 4
archivo['customer_id'].apply(lambda x: len(x)).value_counts()

customer_id
4    114
Name: count, dtype: int64

In [9]:
archivo[archivo.duplicated()] # Hay tres datos repetidos en toda sus filas, lo cual da a entender que es el mismo registro, se
# debe eliminar

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
4,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,2025-12-05,450.50,12,0,Calle Falsa 123
22,C012,Beatriz Pinzón,b.pinzon@eco.com,555-0112,2023-11-15,2025-06-15,850.00,30,0,Av. Jimenez 10
43,C023,Nairo Quintana,n.quintana@ciclismo.co,555-0123,2024-05-20,2025-05-20,99999,50,0,Boyacá Libre


In [10]:
archivo = eliminar_duplicados(archivo, subset=["customer_id"])

In [11]:
archivo

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
0,C001,Juan Perez,jperez@email.com,555-0101,2023-01-15,2025-12-05,450.50,12,0,Calle Falsa 123
1,C002,Maria Garcia,m.garcia@provider.net,555-0102,2023-02-20,2025-05-10,1200.00,45,0,Carrera 7 # 45-10
2,C003,Carlos Rodriguez,c.rod@work.com,NaN,2023-03-05,2025-03-25,99999,8,1,Av. Siempre Viva 742
3,C004,Ana Martinez,ana.mtz@mail.com,555-0104,2023-04-12,2025-06-01,890.20,22,0,Clle 100 # 15
4,C005,Luis Herrera,lherrera@domain.org,555-0105,2023-05-30,2024-12-20,310.00,5,1,Apt 502 Torre B
...,...,...,...,...,...,...,...,...,...,...
105,C106,Simón Bolivar,s.bolivar@libertador.co,555-0206,2025-01-15,2025-05-15,NaN,100,0,Santa Marta
106,C107,Tirso Duarte,t.duarte@timba.cu,555-0207,2025-02-10,2025-06-10,400.00,15,0,Cali Valle
107,C108,Uriel Henao,u.henao@corrido.co,555-0208,2025-03-05,2025-05-06,600.00,22,0,Boyacá
108,C109,Vasco Nuñez,v.nuñez@balboa.es,555-0209,2025-04-12,2025-06-01,NaN,5,0,Darien


In [12]:
archivo[archivo['customer_id']=='C003'] # Se verifica que se eliminó el duplicado de C003

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
2,C003,Carlos Rodriguez,c.rod@work.com,NaN,2023-03-05,2025-03-25,99999,8,1,Av. Siempre Viva 742


In [13]:
archivo[archivo.duplicated()] # Ya la base no tiene duplicados

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address


# Tratar datos nulos y convertir formato de variables

In [14]:
archivo.isna().sum()

customer_id            0
full_name              0
email                  0
phone                  1
signup_date            0
last_purchase_date     1
monthly_spend         15
total_shipments        2
churn_label            1
home_address           8
dtype: int64

In [15]:
archivo['monthly_spend'] = archivo['monthly_spend'].astype(float) #realmente es un decimal

In [16]:
archivo['total_shipments'] = archivo['total_shipments'].fillna(0) # No hubo envíos
archivo['monthly_spend'] = archivo['monthly_spend'].fillna(0) # No hubo compras

In [17]:
# al analizar el NaN en churn_label se identifica que tenía muchos envíos, pero no registra dirección del hogar, se procede a eliminar
archivo[archivo['churn_label'].isna()]

,customer_id,full_name,email,phone,signup_date,last_purchase_date,monthly_spend,total_shipments,churn_label,home_address
9,C010,Martha Lucía,mlucia@xyz.com,555-0110,2023-10-10,2024-10-10,420.0,1000,NaN,NaN


In [18]:
archivo = archivo[~archivo['churn_label'].isna()] # churn_label es la variable objetivo, no deben quedar NaN
archivo['churn_label'] = archivo['churn_label'].astype(np.int64) # convertir a entero, en realidad es categórica binaria
archivo['total_shipments'] = archivo['total_shipments'].astype(np.int64) # convertir a entero ya que es conteo

In [19]:
archivo["monthly_spend"] = archivo["monthly_spend"].clip(lower=0)

In [20]:
archivo.info()

<class 'pandas.DataFrame'>
Index: 109 entries, 0 to 109
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         109 non-null    str           
 1   full_name           109 non-null    str           
 2   email               109 non-null    str           
 3   phone               108 non-null    str           
 4   signup_date         109 non-null    datetime64[us]
 5   last_purchase_date  108 non-null    datetime64[us]
 6   monthly_spend       109 non-null    float64       
 7   total_shipments     109 non-null    int64         
 8   churn_label         109 non-null    int64         
 9   home_address        102 non-null    str           
dtypes: datetime64[us](2), float64(1), int64(2), str(5)
memory usage: 9.4 KB


# Outliers

In [21]:
features = ["monthly_spend","total_shipments"]
# Detectar outliers con IQR
outliers_iqr = outliers_a_dataframe(detectar_outliers(archivo, features, metodo="IQR"))
# Ver outliers de monthly_spend
outliers_iqr

,index,customer_id,feature,value
0,2,C003,monthly_spend,99999.0
1,22,C023,monthly_spend,99999.0
2,45,C046,monthly_spend,5000.0
3,47,C048,monthly_spend,3500.0
4,53,C054,monthly_spend,10000.0
5,61,C062,monthly_spend,7000.0
6,62,C063,monthly_spend,99999.0
7,64,C065,monthly_spend,5000.0
8,66,C067,monthly_spend,6000.0
9,70,C071,monthly_spend,8000.0


In [22]:
archivo = winsorize_iqr(archivo, "monthly_spend")
archivo = winsorize_iqr(archivo, "total_shipments")

# Anonimización de variables sensibles

Durante el proceso de preparación de datos se mantuvo el anonimato de variables sensibles relacionadas con la identificación de los clientes, tales como el nombre completo, correo electrónico, número telefónico y dirección. Estas variables no fueron utilizadas en el modelado con el fin de proteger la privacidad de los usuarios y evitar posibles riesgos asociados al tratamiento de información personal.

# Exportar archivo

In [23]:
archivo.to_csv('../databases/processed/base_limpia_procesada.csv', sep=',', index=False)